In [3]:
import os
os.environ["DDE_BACKEND"] = "pytorch"

In [10]:
import deepxde as dde
dde.backend.backend_name = "pytorch"
import torch
import numpy as np
import matplotlib.pyplot as plt

PDE 


In [11]:
def pde(x, y):
    dx_dt = dde.grad.jacobian(y, x, i=0)
    dy_dt = dde.grad.jacobian(y, x, i=1)

    eq1 = dx_dt + 2*y[:, 0:1] + y[:, 1:2]
    eq2 = dy_dt + y[:, 0:1] + 2*y[:, 1:2]
    
    return [eq1, eq2]

BOUNDARY CONDITION

In [21]:
def is_on_boundary(X, on_boundary):
    return on_boundary and dde.utils.isclose(X[0], 0)

def boundary_value_x(X):
    return 1

def boundary_value_y(X):
    return 0

ANALYTICAL SOLN

In [22]:
def analytical_x(t):
    return 0.5 * np.exp(-1*t) + 0.5*np.exp(-3*t)

def analytical_y(t):
    return -0.5*np.exp(-1*t) + 0.5*np.exp(-3*t)

def analytical_soln(t):
    x = analytical_x(t)
    y = analytical_y(t)
    return np.hstack((x, y))

TIME DOMAIN AND ICS

In [23]:
domain = dde.geometry.TimeDomain(0, 5)

ic_x = dde.icbc.IC(domain, boundary_value_x, is_on_boundary, component=0)
ic_y = dde.icbc.IC(domain, boundary_value_y, is_on_boundary, component=1)

DATA

In [24]:
data = dde.data.PDE(
    domain,
    pde,
    [ic_x, ic_y],
    num_domain=400,
    num_boundary=2,
    solution = analytical_soln,
    num_test=100
)

In [25]:
layer_size = [1] + [50]*3 + [1]
activation="tanh"
initializer = "Glorot uniform"
net = dde.nn.FNN(layer_size, activation, initializer)
model = dde.Model(data, net)
model.compile("adam", lr=0.001, metrics=["l2 relative error"])

Compiling model...
'compile' took 0.005509 s



In [26]:
loss_history, train_state = model.train(iterations=3000)

Training model...



/media/srimukha-sarma/Windows-SSD/conda-envs/deepxde/lib/python3.11/site-packages/keras/src/initializers/initializers.py:120: UserWarning: The initializer GlorotUniform is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


ValueError: in user code:

    File "/media/srimukha-sarma/Windows-SSD/conda-envs/deepxde/lib/python3.11/site-packages/deepxde/model.py", line 246, in outputs_losses_train  *
        True, inputs, targets, auxiliary_vars, self.data.losses_train
    File "/media/srimukha-sarma/Windows-SSD/conda-envs/deepxde/lib/python3.11/site-packages/deepxde/model.py", line 231, in outputs_losses  *
        losses = losses_fn(targets, outputs_, loss_fn, inputs, self, aux=aux)
    File "/media/srimukha-sarma/Windows-SSD/conda-envs/deepxde/lib/python3.11/site-packages/deepxde/data/data.py", line 13, in losses_train  *
        return self.losses(targets, outputs, loss_fn, inputs, model, aux=aux)
    File "/media/srimukha-sarma/Windows-SSD/conda-envs/deepxde/lib/python3.11/site-packages/deepxde/data/pde.py", line 154, in losses  *
        f = self.pde(inputs, outputs_pde)
    File "/home/srimukha-sarma/pip-tmp/ipykernel_2093569/1339062496.py", line 3, in pde  *
        dy_dt = dde.grad.jacobian(y, x, i=1)

In [ ]:
#Evaluating on custom data
t_test = np.linspace(0, 5, 100)[:, None] #this is for random data
'''
t_test = data.test_x 
'''
#for testing with the test data used in realtime model testing by deepxde
y_test_true = np.hstack((analytical_x(t_test), analytical_y(t_test)))

y_test_pred = model.predict(t_test)

l2_error = dde.metrics.l2_relative_error(y_test_true, y_test_pred)
print(l2_error )